In [1]:

!pip install pyspark -q

import pyspark
print("PySpark version:", pyspark.__version__)

PySpark version: 4.0.2


In [2]:

from google.colab import files
uploaded = files.upload()

Saving patients_data_with_alerts.xlsx to patients_data_with_alerts.xlsx


In [3]:

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os


df = pd.read_excel("patients_data_with_alerts.xlsx")


df = df[["Patient Number", "Heart Rate (bpm)"]].copy()
df.columns = ["patient_id", "heart_rate"]


base_time = datetime(2024, 1, 1, 8, 0, 0)
df["timestamp"] = [base_time + timedelta(seconds=30*i) for i in range(len(df))]


os.makedirs("/content/iomt_data", exist_ok=True)
df.to_csv("/content/iomt_data/patients.csv", index=False)

print(f"Rows: {len(df)}")
print(df.head())
print("\nTime range:", df["timestamp"].min(), "→", df["timestamp"].max())

Rows: 50000
   patient_id  heart_rate           timestamp
0           1          96 2024-01-01 08:00:00
1           2          76 2024-01-01 08:00:30
2           3          92 2024-01-01 08:01:00
3           4         137 2024-01-01 08:01:30
4           5          76 2024-01-01 08:02:00

Time range: 2024-01-01 08:00:00 → 2024-01-18 16:39:30


In [4]:

print("Total rows:", len(df))
print("Unique patients:", df["patient_id"].nunique())
print("Heart rate range:", df["heart_rate"].min(), "→", df["heart_rate"].max())
print("Any nulls:", df.isnull().sum().to_dict())

Total rows: 50000
Unique patients: 50000
Heart rate range: 60 → 149
Any nulls: {'patient_id': 0, 'heart_rate': 0, 'timestamp': 0}


In [5]:

import os
import pandas as pd


df = pd.read_csv("/content/iomt_data/patients.csv")


os.makedirs("/content/stream_input", exist_ok=True)
os.makedirs("/content/stream_checkpoint", exist_ok=True)


batch_size = 100
batches = [df.iloc[i:i+batch_size] for i in range(0, len(df), batch_size)]


batches[0].to_csv("/content/stream_input/batch_000.csv", index=False)


os.makedirs("/content/stream_staging", exist_ok=True)
for idx, batch in enumerate(batches[1:], start=1):
    batch.to_csv(f"/content/stream_staging/batch_{idx:03d}.csv", index=False)

print(f"Total batches: {len(batches)}")
print(f"Batch 0 dropped into stream_input ✅")
print(f"Remaining {len(batches)-1} batches staged and ready")

Total batches: 500
Batch 0 dropped into stream_input ✅
Remaining 499 batches staged and ready


In [6]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import window, avg, col
from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType


spark = SparkSession.builder \
    .appName("ICU_HeartRate_Monitor") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark session started ✅")


schema = StructType([
    StructField("patient_id", IntegerType(), True),
    StructField("heart_rate", IntegerType(), True),
    StructField("timestamp", TimestampType(), True)
])


raw_stream = spark.readStream \
    .schema(schema) \
    .option("header", "true") \
    .option("maxFilesPerTrigger", 1) \
    .csv("/content/stream_input")


windowed = raw_stream \
    .withWatermark("timestamp", "2 minutes") \
    .groupBy(
        col("patient_id"),
        window(col("timestamp"), "2 minutes")
    ) \
    .agg(avg("heart_rate").alias("avg_heart_rate"))


alerts = windowed.filter(col("avg_heart_rate") > 100) \
    .select(
        col("patient_id"),
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("avg_heart_rate")
    )


query = alerts.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .option("checkpointLocation", "/content/stream_checkpoint") \
    .trigger(processingTime="10 seconds") \
    .start()

print("Streaming query started ✅")
print("Waiting for first batch to process...")
query.awaitTermination(timeout=30)

Spark session started ✅
Streaming query started ✅
Waiting for first batch to process...


False

In [7]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import window, avg, col
from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType

spark = SparkSession.builder \
    .appName("ICU_HeartRate_Monitor") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

schema = StructType([
    StructField("patient_id", IntegerType(), True),
    StructField("heart_rate", IntegerType(), True),
    StructField("timestamp", TimestampType(), True)
])

raw_stream = spark.readStream \
    .schema(schema) \
    .option("header", "true") \
    .option("maxFilesPerTrigger", 1) \
    .csv("/content/stream_input")

windowed = raw_stream \
    .withWatermark("timestamp", "2 minutes") \
    .groupBy(
        col("patient_id"),
        window(col("timestamp"), "2 minutes")
    ) \
    .agg(avg("heart_rate").alias("avg_heart_rate"))

alerts = windowed.filter(col("avg_heart_rate") > 100) \
    .select(
        col("patient_id"),
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("avg_heart_rate")
    )

query = alerts.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .option("checkpointLocation", "/content/stream_checkpoint2") \
    .trigger(processingTime="10 seconds") \
    .start()

print("Stream restarted ✅")

Stream restarted ✅


In [8]:

import time
import os

for i in range(1, 21):
    src = f"/content/stream_staging/batch_{i:03d}.csv"
    dst = f"/content/stream_input/batch_{i:03d}.csv"
    if os.path.exists(src):
        os.rename(src, dst)
        print(f"Dropped batch_{i:03d}.csv")
        time.sleep(8)

print("\nAll batches dropped ✅")

query.awaitTermination(timeout=60)

Dropped batch_001.csv
Dropped batch_002.csv
Dropped batch_003.csv
Dropped batch_004.csv
Dropped batch_005.csv
Dropped batch_006.csv
Dropped batch_007.csv
Dropped batch_008.csv
Dropped batch_009.csv
Dropped batch_010.csv
Dropped batch_011.csv
Dropped batch_012.csv
Dropped batch_013.csv
Dropped batch_014.csv
Dropped batch_015.csv
Dropped batch_016.csv
Dropped batch_017.csv
Dropped batch_018.csv
Dropped batch_019.csv
Dropped batch_020.csv

All batches dropped ✅


False

In [9]:

query.stop()
print("Previous query stopped ✅")


import shutil
shutil.rmtree("/content/stream_checkpoint2", ignore_errors=True)
shutil.rmtree("/content/stream_checkpoint3", ignore_errors=True)


query2 = alerts.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("hr_alerts") \
    .option("checkpointLocation", "/content/stream_checkpoint3") \
    .trigger(processingTime="10 seconds") \
    .start()

print("Stream with memory sink started ✅")


import time, os
for i in range(21, 36):
    src = f"/content/stream_staging/batch_{i:03d}.csv"
    dst = f"/content/stream_input/batch_{i:03d}.csv"
    if os.path.exists(src):
        os.rename(src, dst)
        print(f"Dropped batch_{i:03d}.csv")
        time.sleep(5)


time.sleep(15)


print("\n--- ALERT OUTPUT ---")
spark.sql("SELECT * FROM hr_alerts ORDER BY avg_heart_rate DESC").show(20, truncate=False)

Previous query stopped ✅
Stream with memory sink started ✅
Dropped batch_021.csv
Dropped batch_022.csv
Dropped batch_023.csv
Dropped batch_024.csv
Dropped batch_025.csv
Dropped batch_026.csv
Dropped batch_027.csv
Dropped batch_028.csv
Dropped batch_029.csv
Dropped batch_030.csv
Dropped batch_031.csv
Dropped batch_032.csv
Dropped batch_033.csv
Dropped batch_034.csv
Dropped batch_035.csv

--- ALERT OUTPUT ---
+----------+-------------------+-------------------+--------------+
|patient_id|window_start       |window_end         |avg_heart_rate|
+----------+-------------------+-------------------+--------------+
|524       |2024-01-01 12:20:00|2024-01-01 12:22:00|149.0         |
|560       |2024-01-01 12:38:00|2024-01-01 12:40:00|149.0         |
|437       |2024-01-01 11:38:00|2024-01-01 11:40:00|149.0         |
|374       |2024-01-01 11:06:00|2024-01-01 11:08:00|149.0         |
|1005      |2024-01-01 16:22:00|2024-01-01 16:24:00|149.0         |
|222       |2024-01-01 09:50:00|2024-01-01 09

In [10]:

print("=" * 70)
print("  ICU HEART RATE MONITORING — CLINICAL ALERTS")
print("  Scenario B: Tumbling 2-min Window | Threshold: avg HR > 100 bpm")
print("=" * 70)

spark.sql("""
    SELECT
        patient_id,
        window_start,
        window_end,
        ROUND(avg_heart_rate, 1) as avg_heart_rate_bpm,
        'CLINICAL ALERT' as alert_status
    FROM hr_alerts
    ORDER BY avg_heart_rate DESC
""").show(20, truncate=False)

print("=" * 70)
print(f"Total alerts fired: {spark.sql('SELECT COUNT(*) FROM hr_alerts').collect()[0][0]}")
print("=" * 70)

  ICU HEART RATE MONITORING — CLINICAL ALERTS
  Scenario B: Tumbling 2-min Window | Threshold: avg HR > 100 bpm
+----------+-------------------+-------------------+------------------+--------------+
|patient_id|window_start       |window_end         |avg_heart_rate_bpm|alert_status  |
+----------+-------------------+-------------------+------------------+--------------+
|1143      |2024-01-01 17:30:00|2024-01-01 17:32:00|149.0             |CLINICAL ALERT|
|925       |2024-01-01 15:42:00|2024-01-01 15:44:00|149.0             |CLINICAL ALERT|
|969       |2024-01-01 16:04:00|2024-01-01 16:06:00|149.0             |CLINICAL ALERT|
|1268      |2024-01-01 18:32:00|2024-01-01 18:34:00|149.0             |CLINICAL ALERT|
|560       |2024-01-01 12:38:00|2024-01-01 12:40:00|149.0             |CLINICAL ALERT|
|524       |2024-01-01 12:20:00|2024-01-01 12:22:00|149.0             |CLINICAL ALERT|
|374       |2024-01-01 11:06:00|2024-01-01 11:08:00|149.0             |CLINICAL ALERT|
|437       |2024-0